# PySpark Simple Tutorial

Clean and minimal PySpark operations with PostgreSQL integration.

## Environment Setup and Imports

In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
# from datetime import date

load_dotenv()

True

## SparkSession Configuration

In [2]:
spark = SparkSession.builder \
    .appName("PySpark PostgreSQL") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/Users/eva/Documents/Projects/Fruta/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/eva/.ivy2.5.2/cache
The jars for the packages stored in: /Users/eva/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e3cef88a-a7e2-4a59-9666-b221247120f4;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 91ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	

## Database Connection Setup

In [3]:
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

jdbc_url = f"jdbc:postgresql://{db_host}:{db_port}/{db_name}"

## Loading Data from PostgreSQL

In [4]:
# JDBC connection properties
jdbc_props = {
    "user": db_user,
    "password": db_password,
    "driver": "org.postgresql.Driver"
}

def show_tables():
    query = "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'"
    return spark.read.jdbc(jdbc_url, f"({query}) as t", properties=jdbc_props)

def load_table(table_name, limit=None):
    query = f"SELECT * FROM {table_name}"
    if limit:
        query += f" LIMIT {limit}"
    return spark.read.jdbc(jdbc_url, f"({query}) as t", properties=jdbc_props)

tables = show_tables()
tables.show()

+----------+
|table_name|
+----------+
|  profiles|
| scenarios|
|   records|
|     notes|
+----------+



In [5]:
dfs_records = load_table("records", limit=100)
dfs_records.show(5)

+-------+---------+-----------+--------------------+--------------------+--------------------+--------------------+
|note_id|client_id|scenario_id|                date|                note|      te3s_embedding|       nfi_embedding|
+-------+---------+-----------+--------------------+--------------------+--------------------+--------------------+
|nn_1332|    cn_05|     sn_056|2024-05-20 19:50:...|Aan het einde van...|[0.02407075,0.043...|[-0.2394495,0.438...|
|nn_1333|    cn_05|     sn_056|2024-05-21 10:00:...|Kleedhulp begonne...|[0.01786319,0.017...|[0.23683387,-0.03...|
|nn_1334|    cn_05|     sn_056|2024-05-21 16:15:...|Extra rusttijd ge...|[0.0023010331,0.0...|[-0.048018847,-0....|
|nn_1335|    cn_05|     sn_056|2024-05-21 22:30:...|Aan het einde van...|[0.027104579,0.02...|[-0.53555995,0.24...|
|nn_1336|    cn_05|     sn_056|2024-05-22 07:50:...|Voor het ontbijt ...|[0.015890764,0.00...|[0.26444602,-0.25...|
+-------+---------+-----------+--------------------+--------------------

In [8]:
# Create sample DataFrame for demonstration
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("salary", DoubleType(), True)
])

data = [
    ("Alice", 28, "Amsterdam", 65000.0),
    ("Bob", 34, "Rotterdam", 72000.0),
    ("Charlie", 29, "Utrecht", 58000.0),
    ("Diana", 31, "Den Haag", 69000.0)
]

df = spark.createDataFrame(data, schema)

## DataFrame Operations

In [9]:
# Basic operations
df.show()
df.printSchema()
df.count()

+-------+---+---------+-------+
|   name|age|     city| salary|
+-------+---+---------+-------+
|  Alice| 28|Amsterdam|65000.0|
|    Bob| 34|Rotterdam|72000.0|
|Charlie| 29|  Utrecht|58000.0|
|  Diana| 31| Den Haag|69000.0|
+-------+---+---------+-------+

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)



4

In [10]:
# Select columns
df.select("name", "salary").show()

# Filter rows
df.filter(col("salary") > 60000).show()

# Add new column
df.withColumn("monthly_salary", col("salary") / 12).show()

# Sort data
df.orderBy(col("salary").desc()).show()

+-------+-------+
|   name| salary|
+-------+-------+
|  Alice|65000.0|
|    Bob|72000.0|
|Charlie|58000.0|
|  Diana|69000.0|
+-------+-------+

+-----+---+---------+-------+
| name|age|     city| salary|
+-----+---+---------+-------+
|Alice| 28|Amsterdam|65000.0|
|  Bob| 34|Rotterdam|72000.0|
|Diana| 31| Den Haag|69000.0|
+-----+---+---------+-------+

+-------+---+---------+-------+-----------------+
|   name|age|     city| salary|   monthly_salary|
+-------+---+---------+-------+-----------------+
|  Alice| 28|Amsterdam|65000.0|5416.666666666667|
|    Bob| 34|Rotterdam|72000.0|           6000.0|
|Charlie| 29|  Utrecht|58000.0|4833.333333333333|
|  Diana| 31| Den Haag|69000.0|           5750.0|
+-------+---+---------+-------+-----------------+

+-------+---+---------+-------+
|   name|age|     city| salary|
+-------+---+---------+-------+
|    Bob| 34|Rotterdam|72000.0|
|  Diana| 31| Den Haag|69000.0|
|  Alice| 28|Amsterdam|65000.0|
|Charlie| 29|  Utrecht|58000.0|
+-------+---+------

## SQL Queries

In [11]:
df.createOrReplaceTempView("employees")

# Simple SQL query
spark.sql("SELECT name, city, salary FROM employees WHERE salary > 60000").show()

# Aggregation query
spark.sql("""
    SELECT city, 
           COUNT(*) as count,
           AVG(salary) as avg_salary
    FROM employees 
    GROUP BY city
""").show()

+-----+---------+-------+
| name|     city| salary|
+-----+---------+-------+
|Alice|Amsterdam|65000.0|
|  Bob|Rotterdam|72000.0|
|Diana| Den Haag|69000.0|
+-----+---------+-------+

+---------+-----+----------+
|     city|count|avg_salary|
+---------+-----+----------+
|Amsterdam|    1|   65000.0|
|Rotterdam|    1|   72000.0|
|  Utrecht|    1|   58000.0|
| Den Haag|    1|   69000.0|
+---------+-----+----------+



## Aggregations and Grouping

In [12]:
# Simple aggregations
df.agg(
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary"),
    count("*").alias("total_count")
).show()

# Group by operations
df.groupBy("city").agg(
    count("*").alias("count"),
    avg("salary").alias("avg_salary")
).show()

+----------+----------+-----------+
|avg_salary|max_salary|total_count|
+----------+----------+-----------+
|   66000.0|   72000.0|          4|
+----------+----------+-----------+

+---------+-----+----------+
|     city|count|avg_salary|
+---------+-----+----------+
|Amsterdam|    1|   65000.0|
|Rotterdam|    1|   72000.0|
|  Utrecht|    1|   58000.0|
| Den Haag|    1|   69000.0|
+---------+-----+----------+



## Data Export

In [14]:
output_dir = "/Users/eva/Documents/Projects/Fruta/output"

# Export to CSV
df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{output_dir}/employees.csv")

# Export to JSON
df.coalesce(1).write.mode("overwrite").json(f"{output_dir}/employees.json")

# Export to Parquet
df.write.mode("overwrite").parquet(f"{output_dir}/employees.parquet")

# Convert to Pandas
pandas_df = df.toPandas()

## Resource Cleanup

In [15]:
# Clean up resources
df.unpersist()
spark.stop()